# Notebook 02 — Territory Performance

**Project:** APAC Territory Planning  
**Objective:** Measure revenue and pipeline performance by territory — quota attainment, coverage ratio, win rate, and geographic distribution via choropleth maps.

**Business Question:** Are APAC territories balanced by opportunity?

In [1]:
import pandas as pd
import duckdb
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA_DIR = "../data/raw"

In [2]:
accounts      = pd.read_csv(f"{DATA_DIR}/accounts.csv")
reps          = pd.read_csv(f"{DATA_DIR}/reps.csv")
assignments   = pd.read_csv(f"{DATA_DIR}/assignments.csv")
opportunities = pd.read_csv(f"{DATA_DIR}/opportunities.csv")
customers     = pd.read_csv(f"{DATA_DIR}/customers.csv")

con = duckdb.connect()
con.register("accounts",      accounts)
con.register("reps",          reps)
con.register("assignments",   assignments)
con.register("opportunities", opportunities)
con.register("customers",     customers)

print("Tables registered:")
con.execute("SHOW TABLES").df()

Tables registered:


,name
0,accounts
1,assignments
2,customers
3,opportunities
4,reps


## 1. Quota Attainment by Rep

In [3]:
sql = """
SELECT
    r.rep_id,
    r.rep_name,
    r.subregion,
    r.segment_focus,
    r.quota_usd,
    COALESCE(SUM(c.arr), 0)                                AS actual_arr,
    COALESCE(SUM(c.arr), 0) / r.quota_usd * 100            AS quota_attainment_pct,
    COUNT(DISTINCT c.customer_id)                          AS n_customers
FROM reps r
LEFT JOIN customers c ON r.rep_id = c.rep_id
WHERE r.subregion != 'Regional'
GROUP BY r.rep_id, r.rep_name, r.subregion, r.segment_focus, r.quota_usd
ORDER BY quota_attainment_pct DESC
"""

os.makedirs("../sql", exist_ok=True)
with open("../sql/quota_attainment.sql", "w") as f:
    f.write(sql)

quota = con.execute(sql).df()

print("QUOTA ATTAINMENT BY REP")
print(f"{'Rep':<22} {'Subregion':<16} {'Segment':<12} {'Quota':>10} {'Actual ARR':>11} {'Attain%':>8} {'Custs':>6}")
print("─" * 90)
for _, row in quota.iterrows():
    print(f"{row['rep_name']:<22} {row['subregion']:<16} {row['segment_focus']:<12} "
          f"{row['quota_usd']:>10,.0f} {row['actual_arr']:>11,.0f} "
          f"{row['quota_attainment_pct']:>7.1f}% {row['n_customers']:>6,.0f}")

QUOTA ATTAINMENT BY REP
Rep                    Subregion        Segment           Quota  Actual ARR  Attain%  Custs
──────────────────────────────────────────────────────────────────────────────────────────
Shannon Gilmore        Greater China    SMB             400,000   2,356,200   589.1%     31
Kristen Martinez       SEA              SMB             400,000   2,040,000   510.0%     46
Toni Ortiz             ANZ              Enterprise    1,200,000   4,752,300   396.0%     83
William Carpenter      ANZ              Enterprise    1,200,000   4,020,300   335.0%     82
Donald Murphy          SEA              SMB             400,000   1,281,000   320.2%     49
Kyle Swanson           North Asia       Mid-Market      809,000   2,543,000   314.3%     97
Jennifer Brooks        Greater China    Mid-Market      700,000   1,922,400   274.6%     36
Christine Pineda       Greater China    Mid-Market      700,000   1,756,800   251.0%     36
Sharon Marshall        North Asia       Enterprise    1,4

## 2. Pipeline Coverage Ratio by Subregion

In [4]:
sql = """
WITH rep_quotas AS (
    SELECT
        subregion,
        SUM(quota_usd) AS total_quota
    FROM reps
    WHERE subregion != 'Regional'
    GROUP BY subregion
),
opp_pipeline AS (
    SELECT
        a.subregion,
        COALESCE(SUM(o.arr_potential)
            FILTER (WHERE o.stage NOT IN ('Closed Won','Closed Lost')), 0) AS open_pipeline,
        COUNT(DISTINCT o.opportunity_id)
            FILTER (WHERE o.stage NOT IN ('Closed Won','Closed Lost'))      AS open_opps,
        COUNT(DISTINCT o.opportunity_id)
            FILTER (WHERE o.stage = 'Closed Won')                           AS closed_won,
        COUNT(DISTINCT o.opportunity_id)
            FILTER (WHERE o.stage = 'Closed Lost')                          AS closed_lost
    FROM opportunities o
    JOIN accounts a ON o.account_id = a.account_id
    GROUP BY a.subregion
)
SELECT
    r.subregion,
    r.total_quota,
    p.open_pipeline,
    p.open_pipeline / r.total_quota * 100                  AS coverage_ratio,
    p.open_opps,
    p.closed_won,
    p.closed_lost,
    ROUND(p.closed_won * 100.0 / NULLIF(p.closed_won + p.closed_lost, 0), 1) AS win_rate_pct
FROM rep_quotas r
JOIN opp_pipeline p ON r.subregion = p.subregion
ORDER BY coverage_ratio DESC
"""

with open("../sql/pipeline_coverage.sql", "w") as f:
    f.write(sql)

pipeline = con.execute(sql).df()

print("PIPELINE COVERAGE BY SUBREGION")
print(f"{'Subregion':<18} {'Total Quota':>12} {'Open Pipeline':>14} {'Coverage%':>10} {'Win Rate':>9}")
print("─" * 68)
for _, row in pipeline.iterrows():
    print(f"{row['subregion']:<18} {row['total_quota']:>12,.0f} {row['open_pipeline']:>14,.0f} "
          f"{row['coverage_ratio']:>9.1f}% {row['win_rate_pct']:>8.1f}%")

PIPELINE COVERAGE BY SUBREGION
Subregion           Total Quota  Open Pipeline  Coverage%  Win Rate
────────────────────────────────────────────────────────────────────
Greater China         2,500,000     26,336,000    1053.4%     33.8%
SEA                   4,600,000     25,478,100     553.9%     44.2%
North Asia            3,681,000     20,224,800     549.4%     35.3%
ANZ                   2,400,000     12,547,000     522.8%     28.0%
India                 1,500,000      5,625,900     375.1%     37.2%


## 3. Revenue by Territory — Choropleth Map

In [5]:
sql = """
SELECT
    a.country,
    a.subregion,
    COUNT(DISTINCT c.customer_id)   AS n_customers,
    COALESCE(SUM(c.arr), 0)         AS total_arr,
    COALESCE(AVG(c.arr), 0)         AS avg_arr
FROM accounts a
LEFT JOIN customers c ON a.account_id = c.account_id
GROUP BY a.country, a.subregion
ORDER BY total_arr DESC
"""

with open("../sql/arr_by_country.sql", "w") as f:
    f.write(sql)

arr_country = con.execute(sql).df()

# ISO alpha-3 mapping for choropleth
iso_map = {
    "SG": "SGP", "ID": "IDN", "MY": "MYS", "TH": "THA", "PH": "PHL", "VN": "VNM",
    "CN": "CHN", "HK": "HKG", "TW": "TWN",
    "JP": "JPN", "KR": "KOR",
    "IN": "IND",
    "AU": "AUS", "NZ": "NZL"
}

arr_country["iso_alpha"] = arr_country["country"].map(iso_map)

fig = px.choropleth(
    arr_country,
    locations="iso_alpha",
    color="total_arr",
    hover_name="country",
    hover_data={"n_customers": True, "total_arr": ":,.0f", "avg_arr": ":,.0f", "iso_alpha": False},
    color_continuous_scale="Blues",
    title="Customer ARR by Country",
    scope="asia"
)

fig.update_layout(
    height=550,
    font=dict(size=13),
    coloraxis_colorbar=dict(title="Total ARR (USD)")
)

fig.show()

os.makedirs("../outputs", exist_ok=True)
fig.write_image("../outputs/02_arr_by_country.png", width=900, height=550, scale=2)
print("Saved: ../outputs/02_arr_by_country.png")

Saved: ../outputs/02_arr_by_country.png


In [6]:
print("CUSTOMER ARR BY COUNTRY")
print(f"{'Country':<10} {'Subregion':<18} {'Customers':>10} {'Total ARR':>12} {'Avg ARR':>10}")
print("─" * 55)
for _, row in arr_country.sort_values("total_arr", ascending=False).iterrows():
    print(f"{row['country']:<10} {row['subregion']:<18} {row['n_customers']:>10,.0f} "
          f"{row['total_arr']:>12,.0f} {row['avg_arr']:>10,.0f}")

CUSTOMER ARR BY COUNTRY
Country    Subregion           Customers    Total ARR    Avg ARR
───────────────────────────────────────────────────────
KR         North Asia                150    5,556,700     37,045
AU         ANZ                        89    4,500,600     50,569
NZ         ANZ                        81    4,436,000     54,765
TW         Greater China              63    4,107,600     65,200
JP         North Asia                134    3,453,100     25,769
HK         Greater China              66    3,388,400     51,339
VN         SEA                        68    3,032,900     44,601
CN         Greater China              57    2,858,500     50,149
ID         SEA                        49    1,846,500     37,684
TH         SEA                        51    1,742,000     34,157
IN         India                      93    1,441,400     15,499
PH         SEA                        53    1,391,900     26,262
MY         SEA                        50    1,206,900     24,138
SG        

## 4. Open Pipeline by Country

In [9]:
sql = """
SELECT
    a.country,
    a.subregion,
    COUNT(DISTINCT o.opportunity_id)    AS n_opps,
    COALESCE(SUM(o.arr_potential), 0)   AS open_pipeline
FROM accounts a
LEFT JOIN opportunities o ON a.account_id = o.account_id
    AND o.stage NOT IN ('Closed Won', 'Closed Lost')
GROUP BY a.country, a.subregion
ORDER BY open_pipeline DESC
"""

with open("../sql/pipeline_by_country.sql", "w") as f:
    f.write(sql)

pipeline_country = con.execute(sql).df()
pipeline_country["iso_alpha"] = pipeline_country["country"].map(iso_map)

fig2 = px.choropleth(
    pipeline_country,
    locations="iso_alpha",
    color="open_pipeline",
    hover_name="country",
    hover_data={"n_opps": True, "open_pipeline": ":,.0f", "iso_alpha": False},
    color_continuous_scale="Oranges",
    title="Open Pipeline by Country",
    scope="asia"
)

fig2.update_layout(
    height=550,
    font=dict(size=13),
    coloraxis_colorbar=dict(title="Open Pipeline (USD)")
)

fig2.show()

fig2.write_image("../outputs/02_pipeline_by_country.png", width=900, height=550, scale=2)
print("Saved: ../outputs/02_pipeline_by_country.png")

Saved: ../outputs/02_pipeline_by_country.png


## 5. Quota Attainment by Subregion — Bar Chart

In [11]:
subregion_perf = (
    quota.groupby("subregion")
    .agg(
        total_quota=("quota_usd", "sum"),
        total_arr=("actual_arr", "sum"),
        n_reps=("rep_id", "count")
    )
    .assign(attainment_pct=lambda x: x["total_arr"] / x["total_quota"] * 100)
    .sort_values("attainment_pct", ascending=False)
    .reset_index()
)

max_val = subregion_perf["attainment_pct"].max()

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=subregion_perf["subregion"],
    y=subregion_perf["attainment_pct"],
    text=[f"{v:.0f}%" for v in subregion_perf["attainment_pct"]],
    textposition="outside",
    marker_color=["#2196F3" if v >= 100 else "#FF5722" for v in subregion_perf["attainment_pct"]]
))

fig3.update_layout(
    title="Quota Attainment by Subregion",
    xaxis_title="Subregion",
    yaxis_title="Attainment %",
    yaxis=dict(range=[0, max_val * 1.25]),
    height=500,
    font=dict(size=13),
    showlegend=False
)

fig3.add_hline(
    y=100,
    line_dash="dash",
    line_color="grey",
    annotation_text="100% quota",
    annotation_position="top right"
)

fig3.show()

fig3.write_image("../outputs/02_quota_attainment.png", width=900, height=500, scale=2)
print("Saved: ../outputs/02_quota_attainment.png")

Saved: ../outputs/02_quota_attainment.png


## Key Findings

1. **India is the weakest territory:** Lowest pipeline coverage (375%), lowest quota attainment, lowest avg ARR ($15K) — penetration problem not a capacity problem
2. **Greater China has the biggest upside:** High avg ARR ($55K), decent win rate (34%) but only 57 CN customers — large Enterprise whitespace untouched
3. **ANZ lowest win rate (28%):** Despite being best covered territory, reps are not converting — quality of pipeline or rep focus may be the issue
4. **KR is the ARR anchor:** 150 customers, $5.6M — single biggest country contributor, drives North Asia performance
5. **SEA highest win rate (44%):** Reps converting well despite being overloaded — adding headcount here has clear revenue upside
6. **Quota attainment inflated across all subregions:** Directional only — India is the clear underperformer, ANZ and Greater China lead on relative basis